In [0]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lag, datediff, when, count, countDistinct, coalesce, lit, 
    current_timestamp, date_format, lower, sha2, concat_ws
)
from pyspark.sql.window import Window


def main():
    # Base path for Azure Data Lake Storage Gen2 Gold Container
    GOLD_BASE_PATH = "abfss://gold@healthcaredatatimi.dfs.core.windows.net"

    # Initialize Spark Session
    spark = SparkSession.builder.appName("Healthcare_Gold_ETL_ADF").getOrCreate()

    # Enable Delta Lake Optimizations
    spark.conf.set("spark.sql.shuffle.partitions", "auto")

    # ------------------------------------------------------------------
    # 0. Load Silver Data Sources
    # ------------------------------------------------------------------
    df_patients = spark.table("healthcare_catalog.default.silver_patients")
    df_encounters = spark.table("healthcare_catalog.default.silver_encounters")
    df_conditions = spark.table("healthcare_catalog.default.silver_conditions")
    df_medications = spark.table("healthcare_catalog.default.silver_medication_requests")
    df_procedures = spark.table("healthcare_catalog.default.silver_procedures")

    # ------------------------------------------------------------------
    # 1. Dimension: gold_dim_patient (SCD Type 2 Aware)
    # ------------------------------------------------------------------
    patient_cols = df_patients.columns

    df_gold_dim_patient = (
        df_patients
        .select(
            # Fallback to current_timestamp if valid_from doesn't exist yet
            col("valid_from") if "valid_from" in patient_cols else current_timestamp().alias("valid_from"),
            col("valid_to") if "valid_to" in patient_cols else lit(None).cast("timestamp").alias("valid_to"),
            col("is_current") if "is_current" in patient_cols else lit(True).alias("is_current"),
            col("patient_id"),
            col("gender"),
            col("birth_date"),
            col("age"),
            when(col("age") < 18, "Pediatric")
            .when((col("age") >= 18) & (col("age") < 50), "Adult (18-49)")
            .when((col("age") >= 50) & (col("age") < 65), "Older Adult (50-64)")
            .otherwise("Senior (65+)").alias("age_group"),
            col("marital_status"),
            col("street_address"),
            col("city"),
            col("state"),
            col("postal_code")
        )
        .withColumn(
            "patient_sk",
            sha2(concat_ws("||", col("patient_id"), col("valid_from").cast("string")), 256)
        )
        .withColumn("gold_updated_at", current_timestamp())
    )

    (
        df_gold_dim_patient.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", f"{GOLD_BASE_PATH}/gold_dim_patient")
        .saveAsTable("healthcare_catalog.default.gold_dim_patient")
    )

    # ------------------------------------------------------------------
    # 2. Dimension: gold_dim_condition
    # ------------------------------------------------------------------
    df_gold_dim_condition = (
        df_conditions
        .filter(col("diagnosis_code").isNotNull())
        .select("diagnosis_code", "diagnosis_description")
        .distinct()
        .withColumn(
            "is_chronic_condition",
            when(
                lower(col("diagnosis_description")).rlike("diabetes|heart failure|copd|hypertension|kidney"), 1
            ).otherwise(0)
        )
        .withColumn("gold_updated_at", current_timestamp())
    )
    
    (
        df_gold_dim_condition.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", f"{GOLD_BASE_PATH}/gold_dim_condition")
        .saveAsTable("healthcare_catalog.default.gold_dim_condition")
    )

    # ------------------------------------------------------------------
    # 3. Dimension: gold_dim_date
    # ------------------------------------------------------------------
    date_path = f"{GOLD_BASE_PATH}/gold_dim_date"
    spark.sql(f"""
    CREATE OR REPLACE TABLE healthcare_catalog.default.gold_dim_date 
    USING DELTA
    LOCATION '{date_path}' AS
    SELECT
      CAST(date_format(date_day, 'yyyyMMdd') AS INT) AS date_key,
      date_day AS calendar_date,
      YEAR(date_day) AS year,
      QUARTER(date_day) AS quarter,
      MONTH(date_day) AS month,
      date_format(date_day, 'MMMM') AS month_name,
      DAY(date_day) AS day_of_month,
      date_format(date_day, 'EEEE') AS day_of_week_name,
      CASE WHEN dayofweek(date_day) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
    FROM (
      SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) AS date_day
    )
    """)

    # ------------------------------------------------------------------
    # 4. Consolidated Fact Table: gold_fact_encounters (Point-in-Time Join)
    # ------------------------------------------------------------------
    cond_summary = df_conditions.groupBy("encounter_id").agg(
        countDistinct("diagnosis_code").alias("comorbidity_count"),
        countDistinct(
            when(lower(col("diagnosis_description")).rlike("diabetes|heart failure|copd|hypertension"), col("diagnosis_code"))
        ).alias("high_risk_comorbidity_count")
    )
    meds_summary = df_medications.groupBy("encounter_id").agg(
        count("medication_request_id").alias("prescribed_meds_count")
    )
    procs_summary = df_procedures.groupBy("encounter_id").agg(
        count("procedure_id").alias("procedures_performed_count")
    )

    patient_window = Window.partitionBy("patient_id").orderBy("admission_date")

    df_encounters_windowed = (
        df_encounters
        .withColumn("prev_discharge_date", lag("discharge_date", 1).over(patient_window))
        .withColumn("days_since_last_discharge", datediff(col("admission_date"), col("prev_discharge_date")))
        .withColumn(
            "is_30day_readmission",
            when((col("days_since_last_discharge").isNotNull()) & 
                 (col("days_since_last_discharge") >= 0) & 
                 (col("days_since_last_discharge") <= 30), 1).otherwise(0)
        )
    )

    # Point-in-time join with Gold Dim Patient using admission_date
    df_encounters_with_patient_sk = df_encounters_windowed.join(
        df_gold_dim_patient,
        (df_encounters_windowed.patient_id == df_gold_dim_patient.patient_id) &
        (df_encounters_windowed.admission_date >= df_gold_dim_patient.valid_from) &
        (
            (df_encounters_windowed.admission_date < df_gold_dim_patient.valid_to) |
            (df_gold_dim_patient.valid_to.isNull())
        ),
        "left"
    )

    df_gold_fact_encounters = (
        df_encounters_with_patient_sk
        .join(cond_summary, "encounter_id", "left")
        .join(meds_summary, "encounter_id", "left")
        .join(procs_summary, "encounter_id", "left")
        .select(
            col("encounter_id"),
            col("patient_sk"),  # Ties encounter to the active patient profile version
            df_encounters_windowed["patient_id"],
            date_format(col("admission_date"), "yyyyMMdd").cast("int").alias("admission_date_key"),
            date_format(col("discharge_date"), "yyyyMMdd").cast("int").alias("discharge_date_key"),
            col("admission_date"),
            col("discharge_date"),
            col("length_of_stay_days"),
            col("admission_type_code"),
            col("admission_type"),
            col("discharge_status"),
            col("prev_discharge_date"),
            col("days_since_last_discharge"),
            col("is_30day_readmission"),
            coalesce(col("comorbidity_count"), lit(0)).alias("comorbidity_count"),
            coalesce(col("high_risk_comorbidity_count"), lit(0)).alias("high_risk_comorbidity_count"),
            coalesce(col("prescribed_meds_count"), lit(0)).alias("prescribed_meds_count"),
            coalesce(col("procedures_performed_count"), lit(0)).alias("procedures_performed_count"),
            current_timestamp().alias("gold_updated_at")
        )
    )
    
    (
        df_gold_fact_encounters.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", f"{GOLD_BASE_PATH}/gold_fact_encounters")
        .saveAsTable("healthcare_catalog.default.gold_fact_encounters")
    )

    # ------------------------------------------------------------------
    # 5. Machine Learning Feature Store Table (Latest Patient State)
    # ------------------------------------------------------------------
    # Join features against current active patient state to avoid duplicate feature rows
    df_active_patients = df_gold_dim_patient.filter(col("is_current") == True)

    df_gold_ml_features = (
        df_gold_fact_encounters.alias("f")
        .join(df_active_patients.alias("p"), "patient_id", "inner")
        .select(
            col("f.encounter_id"),
            col("f.patient_id"),
            col("p.age").alias("feature_patient_age"),
            when(col("p.gender") == "M", 1).otherwise(0).alias("feature_gender_is_male"),
            col("p.marital_status").alias("feature_marital_status"),
            col("f.length_of_stay_days").alias("feature_length_of_stay_days"),
            col("f.comorbidity_count").alias("feature_comorbidity_count"),
            col("f.high_risk_comorbidity_count").alias("feature_high_risk_comorbidity_count"),
            col("f.prescribed_meds_count").alias("feature_prescribed_meds_count"),
            col("f.procedures_performed_count").alias("feature_procedures_count"),
            col("f.is_30day_readmission").alias("label_is_30day_readmission"),
            current_timestamp().alias("feature_created_at")
        )
    )
    
    (
        df_gold_ml_features.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", f"{GOLD_BASE_PATH}/gold_ml_readmission_features")
        .saveAsTable("healthcare_catalog.default.gold_ml_readmission_features")
    )


if __name__ == "__main__":
    main()

## **Business Validation**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

spark = SparkSession.builder.getOrCreate()

# Base path for Azure Data Lake Storage Gen2 Gold Container
GOLD_BASE_PATH = "abfss://gold@healthcaredatatimi.dfs.core.windows.net"

# ==============================================================================
# 1. DEFINE TABLE-SPECIFIC BUSINESS VALIDATION RULES
# ==============================================================================

VALIDATION_CONFIG = {
    "healthcare_catalog.default.gold_dim_patient": {
        # Validates natural patient_id and logical demographic rules
        "rules_sql": """
            patient_id IS NOT NULL AND 
            coalesce(age, 0) >= 0 AND 
            coalesce(age, 0) <= 120 AND 
            to_date(birth_date) <= current_date()
        """,
        "quarantine_path": f"{GOLD_BASE_PATH}/gold_dim_patient_quarantine",
        "quarantine_table": "healthcare_catalog.default.gold_dim_patient_quarantine",
        "table_path": f"{GOLD_BASE_PATH}/gold_dim_patient"
    },
    "healthcare_catalog.default.gold_dim_condition": {
        "rules_sql": "diagnosis_code IS NOT NULL AND coalesce(is_chronic_condition, 0) IN (0, 1)",
        "quarantine_path": f"{GOLD_BASE_PATH}/gold_dim_condition_quarantine",
        "quarantine_table": "healthcare_catalog.default.gold_dim_condition_quarantine",
        "table_path": f"{GOLD_BASE_PATH}/gold_dim_condition"
    },
    "healthcare_catalog.default.gold_fact_encounters": {
        # Uses natural patient_id instead of patient_sk + wraps nullable metrics in coalesce
        "rules_sql": """
            patient_id IS NOT NULL AND 
            coalesce(length_of_stay_days, 0) >= 0 AND 
            coalesce(is_30day_readmission, 0) IN (0, 1) AND 
            to_date(discharge_date) >= to_date(admission_date) AND 
            coalesce(comorbidity_count, 0) >= 0 AND 
            coalesce(prescribed_meds_count, 0) >= 0
        """,
        "quarantine_path": f"{GOLD_BASE_PATH}/gold_fact_encounters_quarantine",
        "quarantine_table": "healthcare_catalog.default.gold_fact_encounters_quarantine",
        "table_path": f"{GOLD_BASE_PATH}/gold_fact_encounters"
    },
    "healthcare_catalog.default.gold_ml_readmission_features": {
        "rules_sql": """
            feature_patient_age IS NOT NULL AND 
            coalesce(feature_patient_age, 0) >= 0 AND 
            coalesce(label_is_30day_readmission, 0) IN (0, 1) AND 
            coalesce(feature_length_of_stay_days, 0) >= 0
        """,
        "quarantine_path": f"{GOLD_BASE_PATH}/gold_ml_readmission_features_quarantine",
        "quarantine_table": "healthcare_catalog.default.gold_ml_readmission_features_quarantine",
        "table_path": f"{GOLD_BASE_PATH}/gold_ml_readmission_features"
    }
}

# ==============================================================================
# 2. DEFINE EXPLICIT SCHEMA FOR AUDIT LOGS (Fixes [CANNOT_DETERMINE_TYPE])
# ==============================================================================

AUDIT_SCHEMA = StructType([
    StructField("table_name", StringType(), True),
    StructField("rules_applied", StringType(), True),
    StructField("total_records", LongType(), True),
    StructField("passed_records", LongType(), True),
    StructField("quarantined_records", LongType(), True),
    StructField("pass_rate_pct", DoubleType(), True),
    StructField("validated_at", StringType(), True)  # ISO timestamp string
])

# ==============================================================================
# 3. UNIVERSAL VALIDATION AND QUARANTINE ENGINE
# ==============================================================================

def execute_gold_business_validations():
    audit_logs = []

    print("=================================================================")
    print(" STARTING GOLD STAGE BUSINESS VALIDATIONS ")
    print("=================================================================\n")

    for table_name, config in VALIDATION_CONFIG.items():
        if not spark.catalog.tableExists(table_name):
            print(f"⚠️ Skipping {table_name} - Table does not exist.")
            continue

        # Load full Gold table produced by previous cell
        df_gold = spark.table(table_name)
        total_rows = df_gold.count()

        if total_rows == 0:
            print(f"⚠️ Table {table_name} is empty. Skipping rule evaluation.")
            continue

        # Split into valid and invalid records
        rules_expr = config["rules_sql"]
        df_valid = df_gold.filter(expr(rules_expr))
        df_invalid = df_gold.filter(~expr(rules_expr))

        valid_count = df_valid.count()
        invalid_count = df_invalid.count()
        pass_rate = (valid_count / total_rows) * 100

        print(f"📋 Table: {table_name}")
        print(f"   ├─ Total Rows Scanned : {total_rows}")
        print(f"   ├─ Passed Validation  : {valid_count} ({pass_rate:.2f}%)")
        print(f"   └─ Quarantined Rows   : {invalid_count}\n")

        # 1. Route invalid records to quarantine if bad data exists
        if invalid_count > 0:
            df_quarantined = df_invalid.withColumn("quarantined_at", current_timestamp())
            (
                df_quarantined.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", config["quarantine_path"])
                .saveAsTable(config["quarantine_table"])
            )

            # Clean up main Gold table by overwriting with valid rows only
            (
                df_valid.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option("path", config["table_path"])
                .saveAsTable(table_name)
            )

        # 2. Append Python-native primitive types (avoid PySpark SQL functions in dicts)
        from datetime import datetime
        audit_logs.append((
            str(table_name),
            str(rules_expr.strip()),
            int(total_rows),
            int(valid_count),
            int(invalid_count),
            float(round(pass_rate, 2)),
            datetime.now().isoformat()
        ))

    # Save validation metadata back to Delta for tracking/PowerBI dashboards
    if audit_logs:
        # Passing tuple list with explicit AUDIT_SCHEMA prevents CANNOT_DETERMINE_TYPE
        df_audit = spark.createDataFrame(audit_logs, schema=AUDIT_SCHEMA)
        
        # Cast validated_at string to actual timestamp
        df_audit = df_audit.withColumn("validated_at", col("validated_at").cast(TimestampType()))

        (
            df_audit.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .option("path", f"{GOLD_BASE_PATH}/gold_data_quality_audit")
            .saveAsTable("healthcare_catalog.default.gold_data_quality_audit")
        )
        print("✅ Data Quality Audit records logged to 'gold_data_quality_audit'.")

    print("\n=================================================================")
    print(" ALL GOLD BUSINESS VALIDATIONS COMPLETED SUCCESSFULLY ")
    print("=================================================================")

# Execute validations
execute_gold_business_validations()

 STARTING GOLD STAGE BUSINESS VALIDATIONS 

📋 Table: healthcare_catalog.default.gold_dim_patient
   ├─ Total Rows Scanned : 30
   ├─ Passed Validation  : 30 (100.00%)
   └─ Quarantined Rows   : 0

📋 Table: healthcare_catalog.default.gold_dim_condition
   ├─ Total Rows Scanned : 36
   ├─ Passed Validation  : 36 (100.00%)
   └─ Quarantined Rows   : 0

📋 Table: healthcare_catalog.default.gold_fact_encounters
   ├─ Total Rows Scanned : 568
   ├─ Passed Validation  : 568 (100.00%)
   └─ Quarantined Rows   : 0

📋 Table: healthcare_catalog.default.gold_ml_readmission_features
   ├─ Total Rows Scanned : 1136
   ├─ Passed Validation  : 1136 (100.00%)
   └─ Quarantined Rows   : 0

✅ Data Quality Audit records logged to 'gold_data_quality_audit'.

 ALL GOLD BUSINESS VALIDATIONS COMPLETED SUCCESSFULLY 


In [0]:
%sql
select *
FROM healthcare_catalog.default.gold_dim_patient

valid_from,valid_to,is_current,patient_id,gender,birth_date,age,age_group,marital_status,street_address,city,state,postal_code,patient_sk,gold_updated_at
2026-07-26T21:19:50.105Z,null,true,25cc2755-9643-43a9-837c-f79c04ad8916,female,1997-08-08,28,Adult (18-49),Never Married,802 Pfannerstill Skyway,Westford,Massachusetts,null,155730e6b806ce7e005be08845aaf5e8d12527c156a7527885cb4befa28bb7cd,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,31191928-6acb-4d73-931c-e601cc3a13fa,female,2002-10-24,23,Adult (18-49),Never Married,892 Hoppe Annex,Wakefield,Massachusetts,01880,3357273553baad3055e662904fc5663f4a41edcf5503251524730ef4b751093c,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,346d4b95-5e00-48fe-a21e-076735ca1d74,male,2001-11-29,24,Adult (18-49),Never Married,113 Abshire Heights,Ludlow,Massachusetts,null,477600e9e20aa302751fa163b655da94ee6a9aa0caf22d12f1541cc5f94e8f4f,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,3f8be6b0-15a6-43f4-87f3-1adb737fa598,male,1993-02-17,33,Adult (18-49),Never Married,506 Goldner Parade,Brookline,Massachusetts,02215,410b0e8eb174db74cab15f92d7860a955e6c5424cc285b3ac18d7cb481488042,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,5c818f3d-7051-4b86-8203-1dc624a91804,male,1997-12-26,28,Adult (18-49),Never Married,441 Zemlak Union Unit 91,Milton,Massachusetts,02186,6e8f2960b88ca434970b7b864ce97525ab0b36c84de4d4d9cbe19d70bcd924da,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,5cbc121b-cd71-4428-b8b7-31e53eba8184,male,1945-12-10,80,Senior (65+),S,894 Brakus Bypass,Taunton,Massachusetts,02718,eea95b19f699ba087aeb2a3d195388b9182ae93187fe25728368eb550cce90d1,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,5dc02d13-5c69-4c36-87d5-16738f088300,male,1996-12-04,29,Adult (18-49),Never Married,217 Will Spur Suite 31,Amherst,Massachusetts,null,24db1f188b8b542ea28f4c600db1d4de1c148c1111978ddbdc45e04d3ffe5e3e,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,668605fe-a8dc-4601-ae48-f5bc24bbea74,female,1999-11-14,26,Adult (18-49),Never Married,721 Von Mews Unit 51,Plymouth,Massachusetts,02360,0749cf03d29783bce4941ddacc8808f2ae3ae709c477b85756b564143290fffd,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,67816396-e325-496d-a6ec-c047756b7ce4,male,1999-12-12,26,Adult (18-49),Never Married,638 Brakus Union Suite 44,Northbridge,Massachusetts,null,f99940cfdf81aa81bd281d5bc096c41d132c97422aa3ce8ad4ac0b57a1ed0610,2026-07-28T14:47:02.136Z
2026-07-26T21:19:50.105Z,null,true,6d6fec2a-b149-49c1-b669-4b7106a7aa72,male,1969-12-26,56,Older Adult (50-64),M,176 Walsh Course Suite 77,Worcester,Massachusetts,01545,95a45b24ff0efff47ada811d4bc0042476fbb1a6b8a43a93787cd156aebb427f,2026-07-28T14:47:02.136Z


In [0]:
%sql
SELECT 
  p.age_group,
  p.city,
  COUNT(f.encounter_id) AS total_admissions,
  SUM(f.is_30day_readmission) AS readmission_count,
  ROUND(AVG(f.is_30day_readmission) * 100, 2) AS readmission_rate_pct,
  ROUND(AVG(f.length_of_stay_days), 1) AS avg_length_of_stay,
  ROUND(AVG(f.high_risk_comorbidity_count), 1) AS avg_high_risk_conditions
FROM healthcare_catalog.default.gold_fact_encounters f
JOIN healthcare_catalog.default.gold_dim_patient p ON f.patient_id = p.patient_id
GROUP BY p.age_group, p.city
ORDER BY readmission_rate_pct DESC;

age_group,city,total_admissions,readmission_count,readmission_rate_pct,avg_length_of_stay,avg_high_risk_conditions
Adult (18-49),Milton,276,262,94.93,0.2,0.0
Senior (65+),Taunton,186,132,70.97,0.0,0.0
Senior (65+),Westford,112,78,69.64,6.8,0.0
Adult (18-49),Northbridge,68,38,55.88,0.0,0.0
Adult (18-49),Wakefield,64,34,53.13,0.0,0.0
Adult (18-49),Somerville,72,38,52.78,0.0,0.0
Adult (18-49),Ludlow,78,41,52.56,0.2,0.0
Adult (18-49),Plymouth,29,11,37.93,0.0,0.0
Older Adult (50-64),Hopedale,30,11,36.67,0.0,0.0
Adult (18-49),Boston,4,1,25.0,1.0,0.0
